# The KV Cache: How Transformers Avoid Recomputing the Past

The full code of [The KV Cache: How Transformers Avoid Recomputing the Past](https://residual-stream.lvtan71.workers.dev/posts/kv-cache-explained/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/kv-cache-explained) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/kv-cache-explained.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 5 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q numpy "torch==2.14.*" "transformers==5.19.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "kv-cache-explained"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `toy.py`

The post's 4-token example, decoded with and without a cache.

In [ ]:
!{sys.executable} toy.py

The post printed:

```text
step 1  no-cache=[1.000 1.000]  cached=[1.000 1.000]  rows: 1 vs 1
step 2  no-cache=[0.500 1.000]  cached=[0.500 1.000]  rows: 2 vs 1
step 3  no-cache=[0.497 0.752]  cached=[0.497 0.752]  rows: 3 vs 1
step 4  no-cache=[0.508 0.880]  cached=[0.508 0.880]  rows: 4 vs 1
total K/V rows computed: 10 (= n(n+1)/2)  vs  4 (= n)
step 4: q4 = [1.000 0.500], scores q4 K^T/sqrt(2) = [0.354 0.707 -0.354 0.530], weights = [0.243 0.346 0.120 0.290], output = [0.508 0.880]
```

## `budget.py`

The cache's memory formula for several models, and an idealized bandwidth-bound decode on one H100 for Llama 2 13B.

In [ ]:
!{sys.executable} budget.py

The post printed:

```text
KV cache per token, 16-bit:
  Llama 2 7B, 32 KV heads                512.0 KiB
  Llama 2 70B, GQA, 8 KV heads           320.0 KiB
  Llama 2 70B if it had 64 KV heads     2560.0 KiB
  Qwen2.5-1.5B, GQA, 2 KV heads           28.0 KiB
  DeepSeek-V2, MLA, 512 + 64 cached       67.5 KiB

Llama 2 13B on one H100, 1,024-token contexts, bandwidth-bound decode:
KV heads  batch   weights   cache/seq   memory     latency     throughput
      40      1    26.0 GB     0.84 GB    26.9 GB     8.0 ms/token       125 tokens/s
      40     64    26.0 GB     0.84 GB    79.7 GB    23.8 ms/token     2,689 tokens/s
      40    256    26.0 GB     0.84 GB   240.8 GB    71.9 ms/token     3,562 tokens/s  does not fit in 80 GB
       8      1    22.7 GB     0.17 GB    22.8 GB     6.8 ms/token       147 tokens/s
       8     64    22.7 GB     0.17 GB    33.4 GB    10.0 ms/token     6,417 tokens/s
       8    256    22.7 GB     0.17 GB    65.6 GB    19.6 ms/token    13,068 tokens/s
```

## `real.py`

Qwen2.5-1.5B-Instruct: cached and recomputed logits compared, the cache's measured size, one step's time with and without the cache, and decode time by batch size.

In [ ]:
!{sys.executable} real.py

The post printed:

```text
1. 32 greedy steps after a 256-token prompt, 32-bit: same next token with and without the cache at 32 of 32 steps; largest logit difference 5.6e-05 (logits span about 37)
2. cache size     measured      formula 2 x L x KV x d x 2 bytes x n
     512 tokens      14.0 MiB       14.0 MiB (28 KiB per token)
   2,048 tokens      56.0 MiB       56.0 MiB (28 KiB per token)
   8,192 tokens     224.0 MiB      224.0 MiB (28 KiB per token)
3. one generation step at context n:  with cache   without   ratio
     256 tokens                   45.9 ms    287.1 ms    6.3x
     512 tokens                   47.5 ms    532.1 ms   11.2x
   1,024 tokens                   48.8 ms   1052.7 ms   21.6x
   2,048 tokens                   49.0 ms   2197.0 ms   44.9x
   4,096 tokens                   64.5 ms   4727.3 ms   73.3x
4. batch   one decode step   tokens/s   x batch 1
       1          47.7 ms         21      1.0x
       4          54.4 ms         74      3.5x
      16          99.0 ms        162      7.7x
      64         203.1 ms        315     15.0x
```